# Nifty-Lens — Feature EDA

Exploratory analysis of the `ml_features` table. Goals:

1. Confirm feature distributions match expectations
2. Check target class balance
3. Assess feature-target relationships (do features separate the classes?)
4. Identify redundant or correlated features
5. Surface any data issues before model training

Dataset: 59,713 rows, 50 tickers, 5 years of daily features.

In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text

load_dotenv()
connection_string = (
    f"postgresql://{os.getenv('NEON_USER')}:{os.getenv('NEON_PASSWORD')}"
    f"@{os.getenv('NEON_HOST')}:{os.getenv('NEON_PORT')}"
    f"/{os.getenv('NEON_DATABASE')}?sslmode=require"
)
engine = create_engine(connection_string)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

## 1. Load features

In [ ]:
df = pd.read_sql(
    "SELECT * FROM ml_features WHERE regime IS NOT NULL ORDER BY ticker, date",
    engine,
)
# Cast numeric columns (Postgres returns Decimal objects)
numeric_cols = [
    "ret_1d", "ret_5d", "ret_20d",
    "vol_5d", "vol_20d", "bb_width", "atr_14",
    "rsi_14", "macd", "macd_signal", "macd_diff",
    "volume_ratio", "abs_return_next",
]
for col in numeric_cols:
    df[col] = df[col].astype(float)
df["regime"] = df["regime"].astype(int)

print(f"Rows: {len(df):,}")
print(f"Tickers: {df['ticker'].nunique()}")
print(f"Date range: {df['date'].min()} → {df['date'].max()}")
df.head()

## 2. Feature distributions

Quick histogram grid — each feature should look approximately bell-shaped or reasonably-distributed. Extreme skew or spikes at boundaries would flag data issues.

In [ ]:
feature_cols = [
    "ret_1d", "ret_5d", "ret_20d",
    "vol_5d", "vol_20d", "bb_width", "atr_14",
    "rsi_14", "macd_diff", "volume_ratio",
]

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for ax, col in zip(axes.flat, feature_cols):
    df[col].hist(bins=80, ax=ax, color="steelblue", edgecolor="none")
    ax.set_title(col, fontsize=11)
    ax.grid(True, alpha=0.3)

plt.suptitle("Feature distributions across 59,713 rows", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 3. Target class balance

In [ ]:
regime_counts = df["regime"].value_counts().sort_index()
regime_labels = ["Low (0)", "Medium (1)", "High (2)"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(regime_labels, regime_counts.values, color=["#4a90d9", "#8db365", "#d9534f"])
for bar, val in zip(bars, regime_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 200,
            f"{val:,}\n({val/len(df)*100:.1f}%)",
            ha="center", fontsize=11)

ax.set_ylabel("Count")
ax.set_title("Volatility Regime — Class Balance")
ax.set_ylim(0, regime_counts.max() * 1.15)
plt.tight_layout()
plt.show()

print(f"Class imbalance ratio: {regime_counts.max() / regime_counts.min():.3f}")

## 4. Feature–target relationships

Key question: *do the features actually separate the three volatility regimes?* If distributions overlap completely, the model will struggle.

Boxplots show median and spread of each feature, grouped by regime. Divergent medians suggest the feature is discriminative.

In [ ]:
# Show boxplots of key features by regime
key_features = ["vol_5d", "vol_20d", "atr_14", "bb_width", "rsi_14", "volume_ratio"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, feat in zip(axes.flat, key_features):
    sns.boxplot(
        data=df, x="regime", y=feat,
        ax=ax, palette=["#4a90d9", "#8db365", "#d9534f"],
        showfliers=False,  # hide outliers so the box structure is visible
    )
    ax.set_title(f"{feat} by volatility regime")
    ax.set_xticklabels(["Low", "Medium", "High"])

plt.suptitle("Does each feature separate the regimes?", y=1.01, fontsize=13)
plt.tight_layout()
plt.show();

**Interpretation:**
- `vol_5d`, `vol_20d`, `atr_14`, `bb_width`: should show **strong monotonic separation** (higher vol/atr/bb_width → higher regime). 
- `rsi_14`: likely shows **U-shaped** relationship (extreme RSI values → more volatility)
- `volume_ratio`: elevated in high-regime days (spike-volume events tend to be volatile)

If any feature shows **no separation across regimes**, it's a candidate for removal.

## 5. Feature correlation heatmap

Highly correlated features are redundant — XGBoost handles them fine, but visualizing helps understand feature structure.

In [ ]:
corr = df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
    square=True, cbar_kws={"shrink": 0.8}, ax=ax,
    vmin=-1, vmax=1,
)
ax.set_title("Feature correlation matrix")
plt.tight_layout()
plt.show()

**Expected patterns:**
- `vol_5d` vs `vol_20d`: high correlation (~0.8+), both measure volatility
- `vol_20d` vs `atr_14`: moderate-high (~0.6-0.8), both range-based
- `bb_width` vs `vol_20d`: high (~0.7+), BB width is essentially a vol proxy
- `ret_1d` vs `ret_5d`: moderate (0.3-0.5), overlapping horizons
- Momentum indicators (`rsi_14`, `macd_diff`) should be relatively uncorrelated with vol features

If anything is > 0.95, those features are near-duplicates.

## 6. Single-ticker timeseries view

Plot Reliance's adj_close with regime overlay — a quick visual sanity check that "high regime" days align with visibly turbulent market moves.

In [ ]:
# Load RELIANCE with prices for context
rel = pd.read_sql(
    """
    SELECT f.date, f.vol_20d, f.rsi_14, f.regime, p.adj_close
    FROM ml_features f
    JOIN prices_daily p ON p.ticker = f.ticker AND p.date = f.date
    WHERE f.ticker = 'RELIANCE.NS'
      AND f.regime IS NOT NULL
    ORDER BY f.date
    """,
    engine,
)
rel["date"] = pd.to_datetime(rel["date"])
rel["adj_close"] = rel["adj_close"].astype(float)
rel["vol_20d"] = rel["vol_20d"].astype(float)
rel["regime"] = rel["regime"].astype(int)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top: price with regime shading
axes[0].plot(rel["date"], rel["adj_close"], color="black", linewidth=1)
colors = {0: "#4a90d9", 1: "#8db365", 2: "#d9534f"}
for reg, color in colors.items():
    mask = rel["regime"] == reg
    axes[0].scatter(rel.loc[mask, "date"], rel.loc[mask, "adj_close"],
                    color=color, s=4, alpha=0.4, label=f"Regime {reg}")
axes[0].set_ylabel("RELIANCE adj_close (₹)")
axes[0].set_title("RELIANCE.NS — price colored by volatility regime")
axes[0].legend(loc="upper left", markerscale=3)

# Bottom: rolling vol_20d
axes[1].plot(rel["date"], rel["vol_20d"], color="steelblue", linewidth=1)
axes[1].set_ylabel("20-day annualized vol")
axes[1].set_xlabel("Date")
axes[1].set_title("Rolling 20-day volatility")

plt.tight_layout()
plt.show()

**Visual sanity check:** Red dots (high regime) should cluster around visibly turbulent periods — drawdowns, sharp rallies, and the vol spikes in the bottom panel. If red dots are scattered randomly (including in calm periods), the regime labeling has an issue.

## 7. Findings summary

1. **59,713 feature rows** across 50 tickers with 12 engineered features — clean and balanced
2. **Classes are nearly perfectly balanced** (~33% each) due to per-ticker tertile split; no resampling needed
3. **Strong vol-based separation**: vol_5d, vol_20d, atr_14, bb_width all show clear monotonic relationships with regime. These will be top-importance features.
4. **Momentum features (RSI, MACD) are less separating** but add orthogonal signal — useful for squeezing out the last few points of accuracy
5. **Volume ratio spikes** in high-regime days, consistent with "unusual days = volatile days"
6. **Vol features are moderately correlated** (expected). XGBoost handles this; no features need dropping.


In [ ]:
# Save key plots as PNGs for the repo README
from pathlib import Path

output_dir = Path("../notebooks/plots")
output_dir.mkdir(exist_ok=True, parents=True)

# ---- Class balance ----
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    ["Low", "Medium", "High"],
    regime_counts.values,
    color=["#4a90d9", "#8db365", "#d9534f"],
)
for bar, val in zip(bars, regime_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 150,
            f"{val:,}\n({val/len(df)*100:.1f}%)", ha="center", fontsize=10)
ax.set_ylabel("Count")
ax.set_title("Volatility Regime — Class Balance (per-ticker tertile split)")
plt.tight_layout()
plt.savefig(output_dir / "class_balance.png", dpi=120, bbox_inches="tight")
plt.close()

# ---- Regime-separation boxplots ----
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, feat in zip(axes.flat, key_features):
    sns.boxplot(
        data=df, x="regime", y=feat, ax=ax,
        palette=["#4a90d9", "#8db365", "#d9534f"], showfliers=False,
    )
    ax.set_title(f"{feat} by volatility regime")
    ax.set_xticklabels(["Low", "Medium", "High"])
plt.suptitle("Feature separation across regimes", y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig(output_dir / "regime_separation.png", dpi=120, bbox_inches="tight")
plt.close()

# ---- Correlation heatmap ----
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
    square=True, cbar_kws={"shrink": 0.8}, ax=ax, vmin=-1, vmax=1,
)
ax.set_title("Feature correlation matrix")
plt.tight_layout()
plt.savefig(output_dir / "feature_correlations.png", dpi=120, bbox_inches="tight")
plt.close()

print("✅ Plots saved to notebooks/plots/")